In [18]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

np.random.seed(42)
for i in range(5):
    X[f'random_noise_{i}'] = np.random.normal(loc=0, scale=1, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [19]:
# 1
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

model_tree = DecisionTreeClassifier(random_state=42)
model_tree.fit(X_train, y_train)

print('DecisionTreeClassifier train: ', accuracy_score(model_tree.predict(X_train), y_train))
print('DecisionTreeClassifier test: ', accuracy_score(model_tree.predict(X_test), y_test))

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    oob_score=True,
    random_state=42
)

bagging.fit(X_train, y_train)

print('BaggingClassifier train: ', bagging.oob_score_)
print('BaggingClassifier test: ', accuracy_score(bagging.predict(X_test), y_test))

random_forest = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

random_forest.fit(X_train, y_train)

print('RandomForestClassifier train: ', random_forest.oob_score_)
print('RandomForestClassifier test: ', accuracy_score(random_forest.predict(X_test), y_test))

random_forest_2 = RandomForestClassifier(
    n_estimators=100,
    max_features=15,
    oob_score=True,
    n_jobs=-1,
    random_state=42
)

random_forest_2.fit(X_train, y_train)

print('RandomForestClassifier max_features=15 train: ', random_forest_2.oob_score_)
print('RandomForestClassifier max_features=15 test: ', accuracy_score(random_forest_2.predict(X_test), y_test))

DecisionTreeClassifier train:  1.0
DecisionTreeClassifier test:  0.9035087719298246
BaggingClassifier train:  0.9582417582417583
BaggingClassifier test:  0.9473684210526315
RandomForestClassifier train:  0.9582417582417583
RandomForestClassifier test:  0.9473684210526315
RandomForestClassifier max_features=15 train:  0.9582417582417583
RandomForestClassifier max_features=15 test:  0.956140350877193


In [20]:
# 2
for i in [10, 30, 50, 100, 200, 300]:
    rf = RandomForestClassifier(
        n_estimators=i,
        max_features=15,
        oob_score=True,
        n_jobs=-1,
        random_state=42
    )

    rf.fit(X_train, y_train)

    print(i, rf.oob_score_)
    print(i, accuracy_score(rf.predict(X_test), y_test))

10 0.9230769230769231
10 0.9210526315789473
30 0.9516483516483516
30 0.9473684210526315


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


50 0.9516483516483516
50 0.956140350877193
100 0.9582417582417583
100 0.956140350877193
200 0.9604395604395605
200 0.956140350877193
300 0.9604395604395605
300 0.956140350877193


In [21]:
# 3
from sklearn.inspection import permutation_importance

result = permutation_importance(random_forest_2, X_test, y_test, n_repeats=10, random_state=42)
print(*result['importances_mean'])
table = pd.DataFrame({
    'feature': X_train.columns,
    'importances': random_forest_2.feature_importances_
})
# СОРТИРОВКА ДАТАФРЕЙМА
sorted_table = table.sort_values(by='importances', ascending=False)
sorted_table.head(20)


0.0 0.0026315789473684292 0.0 0.0 0.0 0.0 0.0 0.010526315789473684 0.0 0.0 0.0 0.0 0.0 0.0026315789473684292 0.0 0.0 0.0 0.0 0.0 0.0 0.001754385964912275 0.007894736842105277 -0.0026315789473684405 0.009649122807017541 0.0 0.0 0.0052631578947368585 0.0201754385964912 0.0 0.0 0.0 0.0 0.0 0.0 0.0


,feature,importances
22,worst perimeter,0.218451
20,worst radius,0.180865
7,mean concave points,0.162112
27,worst concave points,0.157047
23,worst area,0.093514
21,worst texture,0.021138
1,mean texture,0.018773
6,mean concavity,0.015221
3,mean area,0.014485
13,area error,0.013771


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'max_features': ['sqrt', 'log2', 0.5, 0.8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    refit=True,
    random_state=42
)

random_search.fit(X_train, y_train)

print(random_search.best_params_)

best_model = random_search.best_estimator_

print(accuracy_score(best_model.predict(X_test), y_test))


{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 5}
0.9473684210526315
